# 语音转写测试 - 印尼语催收对话

本notebook测试使用Faster-Whisper进行印尼语语音转写

## 1. 安装依赖

In [ ]:
# 安装faster-whisper和ffmpeg-python
%pip install faster-whisper ffmpeg-python torchaudio

## 2. 检查音频文件

In [ ]:
import os
from pathlib import Path

data_dir = Path("../data/chat-sample")
audio_files = sorted(list(data_dir.glob("*.mp3")))

print(f"找到 {len(audio_files)} 个音频文件:")
for f in audio_files:
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.2f} MB")

## 3. 用Faster-Whisper转写第一个文件

In [ ]:
from faster_whisper import WhisperModel

# 初始化模型 - 先用medium模型测试，质量可以的话再用large-v3
model_size = "medium"
model = WhisperModel(model_size, device="cpu", compute_type="int8")

print(f"加载模型 {model_size} 完成")

In [ ]:
# 转写第一个文件
audio_file = str(audio_files[0])
print(f"正在转写: {audio_file}")

segments, info = model.transcribe(audio_file, language="id", beam_size=5, word_timestamps=True)

print("检测到的语言: '%s' (概率 %.2f)" % (info.language, info.language_probability))

In [ ]:
# 打印转写结果
print("\n=== 转写结果 ===")
full_text = []
for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))
    full_text.append(segment.text)

print("\n=== 完整文本 ===")
print(" ".join(full_text))

## 4. 保存结果到JSON

In [ ]:
import json

# 重新获取segments并保存
segments, info = model.transcribe(audio_file, language="id", beam_size=5, word_timestamps=True)

transcript_data = {
    "file_name": audio_files[0].name,
    "detected_language": info.language,
    "language_probability": info.language_probability,
    "segments": []
}

for segment in segments:
    segment_data = {
        "start": segment.start,
        "end": segment.end,
        "text": segment.text,
        "words": []
    }
    if segment.words:
        for word in segment.words:
            segment_data["words"].append({
                "word": word.word,
                "start": word.start,
                "end": word.end,
                "probability": word.probability
            })
    transcript_data["segments"].append(segment_data)

# 保存
output_dir = Path("../data/processed/transcripts")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{audio_files[0].stem}.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(transcript_data, f, ensure_ascii=False, indent=2)

print(f"结果已保存到: {output_file}")

## 5. 批量转写所有文件

In [ ]:
def transcribe_file(audio_path: Path, model, language="id"):
    """转写单个音频文件"""
    segments, info = model.transcribe(
        str(audio_path),
        language=language,
        beam_size=5,
        word_timestamps=True
    )
    
    transcript_data = {
        "file_name": audio_path.name,
        "detected_language": info.language,
        "language_probability": info.language_probability,
        "segments": []
    }
    
    # 需要重新迭代，因为segments是生成器
    segments, _ = model.transcribe(
        str(audio_path),
        language=language,
        beam_size=5,
        word_timestamps=True
    )
    
    for segment in segments:
        segment_data = {
            "start": segment.start,
            "end": segment.end,
            "text": segment.text,
            "words": []
        }
        if segment.words:
            for word in segment.words:
                segment_data["words"].append({
                    "word": word.word,
                    "start": word.start,
                    "end": word.end,
                    "probability": word.probability
                })
        transcript_data["segments"].append(segment_data)
    
    return transcript_data

# 批量处理
print(f"开始批量转写 {len(audio_files)} 个文件...")

for audio_file in audio_files:
    print(f"处理: {audio_file.name}...")
    result = transcribe_file(audio_file, model)
    
    output_file = output_dir / f"{audio_file.stem}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"  保存到: {output_file}")

print("\n全部完成!")